# Adaptive Exploration
Original workflow: `GPN_V3.ipynb`. Imports now use the `amgpn` package.
Run from the repository root after an editable install. Configuration keys retain the original notebook/cell identifiers.


In [ ]:
import torch
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#主要变化：使用SiLU取代ReLU,优化模型架构，以此作为新模型迭代起点
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from amgpn.data.wavelet_episodes import *
from amgpn.legacy.v4.adaptive_exploratory import *
import higher
from amgpn.visualization.single import *
from thop import profile, clever_format
#约定：所有训练时使用类数<configured>抽<configured>，总样本<configured>个，每个任务中样本数支持集<configured>，查询集<configured>，
#     所有测试时使用类数<configured>抽<configured>-<configured>，总样本<configured>个，每个任务中样本数支持集<configured>,<configured>,<configured>，查询集<configured>，

# 参数设置
LOSS_FEATURE_DIM = _cfg_require('GPN_V3.ipynb.cell1.module.LOSS_FEATURE_DIM')
N_WAY = _cfg_require('GPN_V3.ipynb.cell1.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3.ipynb.cell1.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3.ipynb.cell1.module.Q_QUERY')
NUM_TASKS = _cfg_require('GPN_V3.ipynb.cell1.module.NUM_TASKS')
EPOCHS = _cfg_require('GPN_V3.ipynb.cell1.module.EPOCHS')
BLR = _cfg_require('GPN_V3.ipynb.cell1.module.BLR')
LOSSLR=_cfg_require('GPN_V3.ipynb.cell1.module.LOSSLR')
MLR = _cfg_require('GPN_V3.ipynb.cell1.module.MLR')
BATCH_SIZE =_cfg_require('GPN_V3.ipynb.cell1.module.BATCH_SIZE')
loss_alpha=_cfg_require('GPN_V3.ipynb.cell1.module.loss_alpha')
SEED = _cfg_require('GPN_V3.ipynb.cell1.module.SEED')
SAVE_MODEL_DIR = Path(_cfg_require('GPN_V3.ipynb.cell1.module.path'))
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SAVE_MODEL_PATH=SAVE_MODEL_DIR/_cfg_require('GPN_V3.ipynb.cell1.module.path__2')
LOAD_MODEL_PATH=_cfg_require('GPN_V3.ipynb.cell1.module.LOAD_MODEL_PATH')
# 数据路径
# current_path=Path(__file__).resolve().parent
# Historical configuration omitted; provide values through private configuration.
dataset_path = _cfg_require('GPN_V3.ipynb.cell1.module.dataset_path')


def main():
    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('GPN_V3.ipynb.cell1.main.train_ratio'),max_sample_count=_cfg_require('GPN_V3.ipynb.cell1.main.max_sample_count'),
                            use_wavelet_denoise=_cfg_require('GPN_V3.ipynb.cell1.main.use_wavelet_denoise'), 
                             wavelet_name=_cfg_require('GPN_V3.ipynb.cell1.main.wavelet_name'), 
                             denoise_level=_cfg_require('GPN_V3.ipynb.cell1.main.denoise_level'),)
                             # Historical configuration omitted; provide values through private configuration.
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3.ipynb.cell1.main.train_ratio__2'),max_sample_count=_cfg_require('GPN_V3.ipynb.cell1.main.max_sample_count__2'),
                            use_wavelet_denoise=_cfg_require('GPN_V3.ipynb.cell1.main.use_wavelet_denoise__2'), 
                             wavelet_name=_cfg_require('GPN_V3.ipynb.cell1.main.wavelet_name__2'), 
                             denoise_level=_cfg_require('GPN_V3.ipynb.cell1.main.denoise_level__2'),)
                             # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    
    # 创建元任务划分集
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
    
    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3.ipynb.cell1.main.num_tasks'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=_cfg_require('GPN_V3.ipynb.cell1.main.batch_size'), shuffle=True, drop_last=True, num_workers=_cfg_require('GPN_V3.ipynb.cell1.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3.ipynb.cell1.main.batch_size__2'), shuffle=True)

    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model = GPN(use_se=True)#GPN_change
    # model = GPN_Optimized(use_repvgg=True, use_convnext=True)#GPN_new
    model = GPN_Optimized(
                 attention_type=_cfg_require('GPN_V3.ipynb.cell1.main.attention_type'),
                 reduction=_cfg_require('GPN_V3.ipynb.cell1.main.reduction'),)

    params = count_parameters(model)
    
    
    x=torch.randn(2,1,_cfg_require('GPN_V3.ipynb.cell1.main.size_or_budget'),_cfg_require('GPN_V3.ipynb.cell1.main.size_or_budget__2'))
    # 前向传播
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values are positive: {(s > 0).all().item()}")

    # 带中间特征输出
    v, s, features = model(x, return_intermediate=True)
    print(f"\nIntermediate features:")
    for name, feat in features.items():
        if isinstance(feat, torch.Tensor):
            print(f"  {name}: {feat.shape}")
    input_tensor = torch.randn(1, 1, _cfg_require('GPN_V3.ipynb.cell1.main.size_or_budget__3'), _cfg_require('GPN_V3.ipynb.cell1.main.size_or_budget__4'))
    
    flops, params = profile(model, inputs=(input_tensor,))
    flops, params = clever_format([flops, params], "%.3f")
    print(f"FLOPs: {flops}, Params: {params}")
    # 创建训练器
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V3.ipynb.cell1.main.use_Mdistance'),use_multi=_cfg_require('GPN_V3.ipynb.cell1.main.use_multi'),save_middle=_cfg_require('GPN_V3.ipynb.cell1.main.save_middle'),
                              prototypes_per_class=_cfg_require('GPN_V3.ipynb.cell1.main.prototypes_per_class'),crop_ratio_h=_cfg_require('GPN_V3.ipynb.cell1.main.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3.ipynb.cell1.main.crop_ratio_l'),resize=_cfg_require('GPN_V3.ipynb.cell1.main.resize'),
                             merge_threshold=_cfg_require('GPN_V3.ipynb.cell1.main.merge_threshold'), merge_start_epoch=_cfg_require('GPN_V3.ipynb.cell1.main.merge_start_epoch'))
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,epochs=EPOCHS,n_ways=N_WAY,save_path=SAVE_MODEL_PATH)
    print("训练完成！")

    # meta_learner.evaluate(test_loader=test_loader,n_ways=N_WAY)
    
def inference():
    '\n    推理示例：加载已保存的模型进行预测\n    \n    !!!若报模型结构错误，检查：<configured>.引用是否为对应模型\n    '
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,
                            train_ratio=_cfg_require('GPN_V3.ipynb.cell1.inference.train_ratio'),max_sample_count=_cfg_require('GPN_V3.ipynb.cell1.inference.max_sample_count'),
                            use_wavelet_denoise=_cfg_require('GPN_V3.ipynb.cell1.inference.use_wavelet_denoise'), 
                            wavelet_name=_cfg_require('GPN_V3.ipynb.cell1.inference.wavelet_name'), 
                            denoise_level=_cfg_require('GPN_V3.ipynb.cell1.inference.denoise_level'),)
                            # Historical configuration omitted; provide values through private configuration.

    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    # model = GPN(use_se=True)
    model = GPN_Optimized(
                 attention_type=_cfg_require('GPN_V3.ipynb.cell1.inference.attention_type'),#se,decoupled,freq_priority
                 reduction=_cfg_require('GPN_V3.ipynb.cell1.inference.reduction'),
                 )
    # 创建trainer并加载模型
    # trainer = GPNTrainer(model, device=device)
    
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V3.ipynb.cell1.inference.use_Mdistance'),use_multi=_cfg_require('GPN_V3.ipynb.cell1.inference.use_multi'),save_middle=_cfg_require('GPN_V3.ipynb.cell1.inference.save_middle'),
                              prototypes_per_class=_cfg_require('GPN_V3.ipynb.cell1.inference.prototypes_per_class'),crop_ratio_h=_cfg_require('GPN_V3.ipynb.cell1.inference.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3.ipynb.cell1.inference.crop_ratio_l'),resize=_cfg_require('GPN_V3.ipynb.cell1.inference.resize'),
                             merge_threshold=_cfg_require('GPN_V3.ipynb.cell1.inference.merge_threshold'), merge_start_epoch=_cfg_require('GPN_V3.ipynb.cell1.inference.merge_start_epoch'))
    meta_learner.load_model(LOAD_MODEL_PATH)
    
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3.ipynb.cell1.inference.num_tasks'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3.ipynb.cell1.inference.batch_size'), shuffle=True)

    # # 调用可视化
    # print("\nStarting t-SNE visualization...")
    # # 使用 test_loader 进行特征提取
    # meta_learner.visualize_tsne(test_loader, 
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    # 评估
    results = meta_learner.full_evaluation(test_dataset, n_trials=_cfg_require('GPN_V3.ipynb.cell1.inference.n_trials'),q_query=_cfg_require('GPN_V3.ipynb.cell1.inference.q_query'))
    plot_all_visualizations(results,output_dir=_cfg_require('GPN_V3.ipynb.cell1.inference.output_dir'))#V2_rnx+rnx_d_ft5:1_L_8_1_5
    print("推理完成")
if __name__ == '__main__':
    main()
    inference()


In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#第一版）不同度量机制测试代码
import torch
import numpy as np 
from amgpn.data.wavelet_episodes import *
from amgpn.legacy.v4.adaptive_exploratory import * 
from amgpn.comparisons.cnn import CNN_Baseline
from amgpn.evaluation.legacy_comparison import * 
from amgpn.visualization.single import *
from amgpn.visualization.prototypes import *

N_WAY = _cfg_require('GPN_V3.ipynb.cell2.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3.ipynb.cell2.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3.ipynb.cell2.module.Q_QUERY')
SEED = _cfg_require('GPN_V3.ipynb.cell2.module.SEED') 
BLR = _cfg_require('GPN_V3.ipynb.cell2.module.BLR')
MAX_PROTO_PER_CLASS=_cfg_require('GPN_V3.ipynb.cell2.module.MAX_PROTO_PER_CLASS')
# Historical configuration omitted; provide values through private configuration.
DATASET_PATH = _cfg_require('GPN_V3.ipynb.cell2.module.DATASET_PATH')
MODEL_DIR = _cfg_require('GPN_V3.ipynb.cell2.module.MODEL_DIR') 
# Historical configuration omitted; provide values through private configuration.

# <configured>. 准备您的MetaDataset
test_dataset=UAVDataset(data_dir_path=DATASET_PATH,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3.ipynb.cell2.module.train_ratio'),max_sample_count=_cfg_require('GPN_V3.ipynb.cell2.module.max_sample_count'),
                        )
meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3.ipynb.cell2.module.num_tasks'))

# <configured>. 准备模型路径
model_paths = _cfg_require('GPN_V3.ipynb.cell2.module.model_paths')

# <configured>. 配置Trainer参数
trainer_configs = [
        {
        'mlr': BLR,
        'use_Mdistance':_cfg_require('GPN_V3.ipynb.cell2.module.use_Mdistance'),
        'use_multi': _cfg_require('GPN_V3.ipynb.cell2.module.use_multi'),      # 是否使用多原型
        'prototypes_per_class':MAX_PROTO_PER_CLASS,
        # 'crop_ratio_h':<configured>,
        # 'crop_ratio_l':<configured>,
        'resize': _cfg_require('GPN_V3.ipynb.cell2.module.resize'),           # 图像尺寸
        'merge_threshold':_cfg_require('GPN_V3.ipynb.cell2.module.merge_threshold'),
    },
    {
        'mlr': BLR,
        # 'crop_ratio_h':<configured>,
        # 'crop_ratio_l':<configured>,
        'resize': _cfg_require('GPN_V3.ipynb.cell2.module.resize__2'),           # 图像尺寸
    },
    {
        'mlr': BLR,
        'use_Mdistance':_cfg_require('GPN_V3.ipynb.cell2.module.use_Mdistance__2'),
        'use_multi': _cfg_require('GPN_V3.ipynb.cell2.module.use_multi__2'),      # 是否使用多原型
        'prototypes_per_class':MAX_PROTO_PER_CLASS,
        # 'crop_ratio_h':<configured>,
        # 'crop_ratio_l':<configured>,
        'resize': _cfg_require('GPN_V3.ipynb.cell2.module.resize__3'),           # 图像尺寸
        'merge_threshold':_cfg_require('GPN_V3.ipynb.cell2.module.merge_threshold__2'),
    },
    {
        'mlr': BLR,
        'use_Mdistance':_cfg_require('GPN_V3.ipynb.cell2.module.use_Mdistance__3'),
        'use_multi': _cfg_require('GPN_V3.ipynb.cell2.module.use_multi__3'),      # 是否使用多原型
        'prototypes_per_class':MAX_PROTO_PER_CLASS,
        # 'crop_ratio_h':<configured>,
        # 'crop_ratio_l':<configured>,
        'resize': _cfg_require('GPN_V3.ipynb.cell2.module.resize__4'),           # 图像尺寸
        'merge_threshold':_cfg_require('GPN_V3.ipynb.cell2.module.merge_threshold__3'),
    },

]

# <configured>. 定义模型初始化函数
model_initializers = [
        lambda: GPN_Optimized(),    
    lambda: CNN_Baseline(reduction=_cfg_require('GPN_V3.ipynb.cell2.module.reduction'), feature_dim=_cfg_require('GPN_V3.ipynb.cell2.module.feature_dim')),
    lambda: GPN_Optimized(),
    lambda: GPN_Optimized(),
    # lambda: GPN_Optimized(),
    # lambda: GPN_Optimized(),
]

#model和device已预设
results_df = test_multiple_models(
    test_dataset=test_dataset,
    model_paths=model_paths,
    trainer_configs=trainer_configs,
    model_initializers=model_initializers,
    n_way=_cfg_require('GPN_V3.ipynb.cell2.module.n_way'),                    # <configured>-way分类
    k_shots=_cfg_require('GPN_V3.ipynb.cell2.module.k_shots'),        # 测试<configured>,<configured>,<configured>,<configured>-shot
    n_trials=_cfg_require('GPN_V3.ipynb.cell2.module.n_trials'),               # 每个配置测试<configured>次
    q_query=_cfg_require('GPN_V3.ipynb.cell2.module.q_query'),               # 查询集大小
    device=_cfg_require('GPN_V3.ipynb.cell2.module.device'),             # 使用GPU
    seed=SEED,                   # 随机种子
    save_dir=_cfg_require('GPN_V3.ipynb.cell2.module.save_dir')       # 结果保存目录
)

# 分析保存的CSV文件
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# 生成额外的可视化
# analyzer.plot_prototype_heatmap()
# analyzer.plot_efficiency_comparison()

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#第一版）不同原型生成机制测试代码
import torch
import numpy as np 
from amgpn.data.wavelet_episodes import *
from amgpn.legacy.v3.fixed_multiprototype import GPN_Optimized as GPN_Kmeans
from amgpn.legacy.v4.fixed_multiprototype import GPN_Optimized as GPN_IMP
from amgpn.legacy.v4.adaptive_exploratory import GPN_Optimized
from amgpn.evaluation.legacy_comparison import * 
from amgpn.visualization.single import *
from amgpn.visualization.prototypes import *

N_WAY = _cfg_require('GPN_V3.ipynb.cell3.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3.ipynb.cell3.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3.ipynb.cell3.module.Q_QUERY')
SEED = _cfg_require('GPN_V3.ipynb.cell3.module.SEED') 
BLR = _cfg_require('GPN_V3.ipynb.cell3.module.BLR')
MAX_PROTO_PER_CLASS=_cfg_require('GPN_V3.ipynb.cell3.module.MAX_PROTO_PER_CLASS')
DATASET_PATH = _cfg_require('GPN_V3.ipynb.cell3.module.DATASET_PATH') 
MODEL_DIR = _cfg_require('GPN_V3.ipynb.cell3.module.MODEL_DIR')
# Historical configuration omitted; provide values through private configuration.

# <configured>. 准备您的MetaDataset
test_dataset=UAVDataset(data_dir_path=DATASET_PATH,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3.ipynb.cell3.module.train_ratio'),max_sample_count=_cfg_require('GPN_V3.ipynb.cell3.module.max_sample_count'),
                        )
meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3.ipynb.cell3.module.num_tasks'))

# <configured>. 准备模型路径
model_paths = _cfg_require('GPN_V3.ipynb.cell3.module.model_paths')

# <configured>. 配置Trainer参数
trainer_configs = [
        {
        'lr': BLR,
        'alpha':_cfg_require('GPN_V3.ipynb.cell3.module.alpha'),
        'sigma_init':_cfg_require('GPN_V3.ipynb.cell3.module.sigma_init'),
        'prototype_mode':_cfg_require('GPN_V3.ipynb.cell3.module.prototype_mode'),
        'crop_ratio_h':_cfg_require('GPN_V3.ipynb.cell3.module.crop_ratio_h'),
        'crop_ratio_l':_cfg_require('GPN_V3.ipynb.cell3.module.crop_ratio_l'),
        'resize': _cfg_require('GPN_V3.ipynb.cell3.module.resize'),           # 图像尺寸
        'distance_type':_cfg_require('GPN_V3.ipynb.cell3.module.distance_type'),
    },
    {
        'mlr': BLR,
        'use_multi': _cfg_require('GPN_V3.ipynb.cell3.module.use_multi'),      # 是否使用多原型
        'crop_ratio_h':_cfg_require('GPN_V3.ipynb.cell3.module.crop_ratio_h__2'),
        'crop_ratio_l':_cfg_require('GPN_V3.ipynb.cell3.module.crop_ratio_l__2'),
        'resize': _cfg_require('GPN_V3.ipynb.cell3.module.resize__2'),           # 图像尺寸
        'prototypes_per_class':_cfg_require('GPN_V3.ipynb.cell3.module.prototypes_per_class'),
    },
    {
        'mlr': BLR,
        'use_multi': _cfg_require('GPN_V3.ipynb.cell3.module.use_multi__2'),      # 是否使用多原型
        'crop_ratio_h':_cfg_require('GPN_V3.ipynb.cell3.module.crop_ratio_h__3'),
        'crop_ratio_l':_cfg_require('GPN_V3.ipynb.cell3.module.crop_ratio_l__3'),
        'resize': _cfg_require('GPN_V3.ipynb.cell3.module.resize__3'),           # 图像尺寸
        'prototypes_per_class':_cfg_require('GPN_V3.ipynb.cell3.module.prototypes_per_class__2'),
    },

    {
        'mlr': BLR,
        'use_Mdistance':_cfg_require('GPN_V3.ipynb.cell3.module.use_Mdistance'),
        'use_multi': _cfg_require('GPN_V3.ipynb.cell3.module.use_multi__3'),      # 是否使用多原型
        'prototypes_per_class':MAX_PROTO_PER_CLASS,
        'crop_ratio_h':_cfg_require('GPN_V3.ipynb.cell3.module.crop_ratio_h__4'),
        'crop_ratio_l':_cfg_require('GPN_V3.ipynb.cell3.module.crop_ratio_l__4'),
        'resize': _cfg_require('GPN_V3.ipynb.cell3.module.resize__4'),           # 图像尺寸
        'merge_threshold':_cfg_require('GPN_V3.ipynb.cell3.module.merge_threshold'),
    },
]

# <configured>. 定义模型初始化函数
model_initializers = [
        lambda: GPN_IMP(),

    lambda: GPN_Kmeans(),
    lambda: GPN_Kmeans(),
    lambda: GPN_Optimized(),
    # lambda: GPN_Optimized(),
    # lambda: GPN_Optimized(),
]

#model和device已预设
results_df = test_multiple_models(
    test_dataset=test_dataset,
    model_paths=model_paths,
    trainer_configs=trainer_configs,
    model_initializers=model_initializers,
    n_way=_cfg_require('GPN_V3.ipynb.cell3.module.n_way'),                    # <configured>-way分类
    k_shots=_cfg_require('GPN_V3.ipynb.cell3.module.k_shots'),        # 测试<configured>,<configured>,<configured>,<configured>-shot
    n_trials=_cfg_require('GPN_V3.ipynb.cell3.module.n_trials'),               # 每个配置测试<configured>次
    q_query=_cfg_require('GPN_V3.ipynb.cell3.module.q_query'),               # 查询集大小
    device=_cfg_require('GPN_V3.ipynb.cell3.module.device'),             # 使用GPU
    seed=SEED,                   # 随机种子
    save_dir=_cfg_require('GPN_V3.ipynb.cell3.module.save_dir')       # 结果保存目录
)

# 分析保存的CSV文件
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# 生成额外的可视化
# analyzer.plot_prototype_heatmap()
# analyzer.plot_efficiency_comparison()

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
from amgpn.visualization.spectrograms import *

if __name__ == '__main__':
    from torch.utils.data import DataLoader
    from amgpn.data.wavelet_episodes import UAVDataset, MetaDataset
    from amgpn.legacy.v4.adaptive_exploratory import GPN_Optimized, GPNTrainer
    
    # 配置参数
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATASET_PATH = _cfg_require('GPN_V3.ipynb.cell4.module.DATASET_PATH')
    MODEL_PATH = _cfg_require('GPN_V3.ipynb.cell4.module.MODEL_PATH')
    SAVE_DIR = _cfg_require('GPN_V3.ipynb.cell4.module.SAVE_DIR')
    
    N_WAY = _cfg_require('GPN_V3.ipynb.cell4.module.N_WAY')
    K_SHOT = _cfg_require('GPN_V3.ipynb.cell4.module.K_SHOT')
    Q_QUERY = _cfg_require('GPN_V3.ipynb.cell4.module.Q_QUERY')
    NUM_SAMPLES = _cfg_require('GPN_V3.ipynb.cell4.module.NUM_SAMPLES')
    SEED = _cfg_require('GPN_V3.ipynb.cell4.module.SEED')
    
    print("加载数据集...")
    test_dataset = UAVDataset(
        data_dir_path=DATASET_PATH,
        seed=SEED,
        is_train=False,
        train_ratio=_cfg_require('GPN_V3.ipynb.cell4.module.train_ratio'),
        max_sample_count=_cfg_require('GPN_V3.ipynb.cell4.module.max_sample_count')
    )
    
    print("创建元任务...")
    meta_test_dataset = MetaDataset(
        test_dataset, 
        n_way=N_WAY, 
        k_shot=K_SHOT, 
        q_query=Q_QUERY,
        num_tasks=_cfg_require('GPN_V3.ipynb.cell4.module.num_tasks')  # 只测试一个任务
    )
    
    test_loader = DataLoader(
        meta_test_dataset, 
        batch_size=_cfg_require('GPN_V3.ipynb.cell4.module.batch_size'), 
        shuffle=False
    )
    
    print("加载模型...")
    model = GPN_Optimized(
                 attention_type=_cfg_require('GPN_V3.ipynb.cell4.module.attention_type'),#se,decoupled,freq_priority
                 reduction=_cfg_require('GPN_V3.ipynb.cell4.module.reduction'),
                 )
    
    # 加载模型权重
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(DEVICE)
    
    print("开始Grad-CAM分析...")
    test_gradcam_analysis(
        model=model,
        test_loader=test_loader,
        device=DEVICE,
        save_dir=SAVE_DIR,
        n_way=N_WAY,
        k_shot=K_SHOT,
        q_query=Q_QUERY,
        num_samples=NUM_SAMPLES,
        use_multi_prototype=_cfg_require('GPN_V3.ipynb.cell4.module.use_multi_prototype'),  # 根据训练配置设置
        prototypes_per_class=_cfg_require('GPN_V3.ipynb.cell4.module.prototypes_per_class'),     # 与训练时保持一致
        crop_ratio=_cfg_require('GPN_V3.ipynb.cell4.module.crop_ratio')
    )
    
    print("\n分析完成!")

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#不同度量机制
import torch
import numpy as np 
from amgpn.data.wavelet_episodes import *
from amgpn.legacy.v4.adaptive_exploratory import * 
from amgpn.evaluation.legacy_comparison import * 
from amgpn.visualization.single import *
from amgpn.visualization.prototypes import *

N_WAY = _cfg_require('GPN_V3.ipynb.cell5.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3.ipynb.cell5.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3.ipynb.cell5.module.Q_QUERY')
SEED = _cfg_require('GPN_V3.ipynb.cell5.module.SEED') 
BLR = _cfg_require('GPN_V3.ipynb.cell5.module.BLR')
MAX_PROTO_PER_CLASS=_cfg_require('GPN_V3.ipynb.cell5.module.MAX_PROTO_PER_CLASS')
DATASET_PATH = _cfg_require('GPN_V3.ipynb.cell5.module.DATASET_PATH') 
MODEL_DIR = _cfg_require('GPN_V3.ipynb.cell5.module.MODEL_DIR') 
# Historical configuration omitted; provide values through private configuration.

# <configured>. 准备您的MetaDataset
test_dataset=UAVDataset(data_dir_path=DATASET_PATH,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3.ipynb.cell5.module.train_ratio'),max_sample_count=_cfg_require('GPN_V3.ipynb.cell5.module.max_sample_count'),
                        )
meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3.ipynb.cell5.module.num_tasks'))

# <configured>. 准备模型路径
model_paths = _cfg_require('GPN_V3.ipynb.cell5.module.model_paths')

# <configured>. 配置Trainer参数
trainer_configs = [
    {
        'mlr': BLR,
        'use_Mdistance':_cfg_require('GPN_V3.ipynb.cell5.module.use_Mdistance'),
        'use_multi': _cfg_require('GPN_V3.ipynb.cell5.module.use_multi'),      # 是否使用多原型
        'prototypes_per_class':MAX_PROTO_PER_CLASS,
        'crop_ratio_h':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_h'),
        'crop_ratio_l':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_l'),
        'resize': _cfg_require('GPN_V3.ipynb.cell5.module.resize'),           # 图像尺寸
        'merge_threshold':_cfg_require('GPN_V3.ipynb.cell5.module.merge_threshold'),
    },    {
        'mlr': BLR,
        'use_Mdistance':_cfg_require('GPN_V3.ipynb.cell5.module.use_Mdistance__2'),
        'use_multi': _cfg_require('GPN_V3.ipynb.cell5.module.use_multi__2'),      # 是否使用多原型
        'prototypes_per_class':MAX_PROTO_PER_CLASS,
        'crop_ratio_h':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_h__2'),
        'crop_ratio_l':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_l__2'),
        'resize': _cfg_require('GPN_V3.ipynb.cell5.module.resize__2'),           # 图像尺寸
        'merge_threshold':_cfg_require('GPN_V3.ipynb.cell5.module.merge_threshold__2'),
    },    {
        'mlr': BLR,
        'use_Mdistance':_cfg_require('GPN_V3.ipynb.cell5.module.use_Mdistance__3'),
        'use_multi': _cfg_require('GPN_V3.ipynb.cell5.module.use_multi__3'),      # 是否使用多原型
        'prototypes_per_class':MAX_PROTO_PER_CLASS,
        'crop_ratio_h':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_h__3'),
        'crop_ratio_l':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_l__3'),
        'resize': _cfg_require('GPN_V3.ipynb.cell5.module.resize__3'),           # 图像尺寸
        'merge_threshold':_cfg_require('GPN_V3.ipynb.cell5.module.merge_threshold__3'),
    },    {
        'mlr': BLR,
        'use_Mdistance':_cfg_require('GPN_V3.ipynb.cell5.module.use_Mdistance__4'),
        'use_multi': _cfg_require('GPN_V3.ipynb.cell5.module.use_multi__4'),      # 是否使用多原型
        'prototypes_per_class':MAX_PROTO_PER_CLASS,
        'crop_ratio_h':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_h__4'),
        'crop_ratio_l':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_l__4'),
        'resize': _cfg_require('GPN_V3.ipynb.cell5.module.resize__4'),           # 图像尺寸
        'merge_threshold':_cfg_require('GPN_V3.ipynb.cell5.module.merge_threshold__4'),
    },
    {
        'mlr': BLR,
        'use_Mdistance':_cfg_require('GPN_V3.ipynb.cell5.module.use_Mdistance__5'),
        'use_multi': _cfg_require('GPN_V3.ipynb.cell5.module.use_multi__5'),      # 是否使用多原型
        'prototypes_per_class':MAX_PROTO_PER_CLASS,
        'crop_ratio_h':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_h__5'),
        'crop_ratio_l':_cfg_require('GPN_V3.ipynb.cell5.module.crop_ratio_l__5'),
        'resize': _cfg_require('GPN_V3.ipynb.cell5.module.resize__5'),           # 图像尺寸
        'merge_threshold':_cfg_require('GPN_V3.ipynb.cell5.module.merge_threshold__5'),
    },
]

# <configured>. 定义模型初始化函数
model_initializers = [
    lambda: GPN_Optimized(),
    lambda: GPN_Optimized(),
    lambda: GPN_Optimized(),
    lambda: GPN_Optimized(),    
    lambda: GPN_Optimized(),
    # lambda: GPN_Optimized(),
]

#model和device已预设
results_df = test_multiple_models(
    test_dataset=test_dataset,
    model_paths=model_paths,
    trainer_configs=trainer_configs,
    model_initializers=model_initializers,
    n_way=_cfg_require('GPN_V3.ipynb.cell5.module.n_way'),                    # <configured>-way分类
    k_shots=_cfg_require('GPN_V3.ipynb.cell5.module.k_shots'),        # 测试<configured>,<configured>,<configured>,<configured>-shot
    n_trials=_cfg_require('GPN_V3.ipynb.cell5.module.n_trials'),               # 每个配置测试<configured>次
    q_query=_cfg_require('GPN_V3.ipynb.cell5.module.q_query'),               # 查询集大小
    device=_cfg_require('GPN_V3.ipynb.cell5.module.device'),             # 使用GPU
    seed=SEED,                   # 随机种子
    save_dir=_cfg_require('GPN_V3.ipynb.cell5.module.save_dir')       # 结果保存目录
)

# 分析保存的CSV文件
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# 生成额外的可视化
# analyzer.plot_prototype_heatmap()
# analyzer.plot_efficiency_comparison()